# SVM L1 & SVM L2 Classification
## Based on ML Notebook 3B: Linear Models for Classification
Organized following the structure of Christopher Monterola, Kenneth Co., and Gino Borja.

**Sections:**
- **Part A — SVM L1:** Single C → Range of C → Monte Carlo Tuning → Best C & Best Feature
- **Part B — SVM L2:** Single C → Range of C → Monte Carlo Tuning → Best C & Best Feature


## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from warnings import simplefilter
from sklearn.exceptions import ConvergenceWarning
simplefilter("ignore", category=ConvergenceWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

pd.options.display.float_format = '{:,.4g}'.format


## 1. Load Data
Change `CSV_PATH` to match your local CSV file. This notebook uses the ML-ready Amazon reviews dataset.


In [ ]:
CSV_PATH   = "/Users/licahc/Downloads/amazon_bestsellers_reviews_train_ml_ready.csv"
TARGET_COL = "is_helpful"

df_raw = pd.read_csv(CSV_PATH)

print(f"Loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(df_raw.dtypes)
df_raw.head()


## 2. Preprocessing
The CSV is already ML-ready, so this section only validates and formats the target column.


In [ ]:
df = df_raw.copy()

TARGET_COL = "is_helpful"

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found.")

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce").fillna(0).astype(int)

print("DataFrame after preprocessing:", df.shape)
print("Target distribution:")
print(df[TARGET_COL].value_counts())

display(
    df[['rating', 'title_len', 'word_count', 'sentiment', 'sentiment_extremity', 'is_helpful']]
    .head()
    .style.set_caption("Main Features Used for SVM Classification")
)


## 3. Feature Selection

In [ ]:
# Use all numeric ML-ready columns except the target column
feature_cols = [
    c for c in df.columns
    if c != TARGET_COL
    and pd.api.types.is_numeric_dtype(df[c])
    and df[c].notna().sum() > 0
]

print("Feature columns selected:")
for c in feature_cols:
    print(f"  {c}")

X_all = df[feature_cols].fillna(0).values
y_all = df[TARGET_COL].values

# Auto-detect unique class labels
unique_classes = np.unique(y_all)
target_names   = [str(c) for c in unique_classes]

print(f"
X shape : {X_all.shape}")
print(f"y shape : {y_all.shape}")
print(f"Classes : {unique_classes}  ->  {target_names}")
print(f"Class balance: {np.bincount(y_all)}")


## 4. Train / Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train size: {X_train_sc.shape[0]:,}  |  Test size: {X_test_sc.shape[0]:,}")


---
# Part A — Support Vector Machine with L1 Regularization (SVM L1)

`LinearSVC(penalty="l1", loss='squared_hinge', dual=False)`

**Higher C = less regularization** (same interpretation as ML Notebook 3B Section D).


### A-1. SVM L1 — Single C (C = 1)

In [ ]:
# SVM L1 with C = 1
C_single_l1 = 1

svm_l1_single = LinearSVC(
    C=C_single_l1,
    penalty="l1",
    loss='squared_hinge',
    dual=False,
    max_iter=10000
).fit(X_train_sc, y_train)

print(f"SVM L1  |  C = {C_single_l1}")
print(f"  Training set score : {svm_l1_single.score(X_train_sc, y_train):.6f}")
print(f"  Test set score     : {svm_l1_single.score(X_test_sc,  y_test):.6f}")


In [ ]:
# Classification Report — SVM L1, C = 1
y_pred_l1_single = svm_l1_single.predict(X_test_sc)
print("Classification Report (SVM L1, C=1):")
print(classification_report(y_test, y_pred_l1_single,
                             labels=unique_classes, target_names=target_names))


In [ ]:
# Confusion Matrix — SVM L1, C = 1
confmat_l1_single = confusion_matrix(y_true=y_test, y_pred=y_pred_l1_single,
                                     labels=unique_classes)

fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.matshow(confmat_l1_single, cmap=plt.cm.Blues, alpha=0.6)
for i in range(confmat_l1_single.shape[0]):
    for j in range(confmat_l1_single.shape[1]):
        ax.text(x=j, y=i, s=confmat_l1_single[i, j], va='center', ha='center', fontsize=14)
tick_labels = [''] + target_names
ax.set_xticklabels(tick_labels, rotation=30, ha='left')
ax.set_yticklabels(tick_labels)
ax.set_xlabel('Predicted label', fontsize=12)
ax.set_ylabel('True label', fontsize=12)
ax.set_title(f'Confusion Matrix — SVM L1  (C={C_single_l1})', fontsize=12, pad=12)
plt.tight_layout()
plt.show()


### A-2. SVM L1 — Range of C Values

In [ ]:
# SVM L1: scores over a range of C values (single split)
C_range_l1 = [1e-4, 1e-3, 0.1, 0.2, 0.4, 0.75, 1, 1.5, 3, 5, 10, 15, 20, 100, 300, 1000, 5000]

train_scores_l1 = []
test_scores_l1  = []

for c in C_range_l1:
    svm = LinearSVC(C=c, penalty="l1", loss='squared_hinge', dual=False, max_iter=10000)
    svm.fit(X_train_sc, y_train)
    train_scores_l1.append(svm.score(X_train_sc, y_train))
    test_scores_l1.append(svm.score(X_test_sc,  y_test))
    print(f"C = {c:>8}  |  train: {train_scores_l1[-1]:.4f}  |  test: {test_scores_l1[-1]:.4f}")


In [ ]:
# Plot — SVM L1 range of C
fig = plt.figure(figsize=(12, 5))
plt.xscale('log')
plt.plot(C_range_l1, train_scores_l1, '-o', label='Training accuracy', color='steelblue')
plt.plot(C_range_l1, test_scores_l1,  '-^', label='Test accuracy',     color='darkorange')
plt.xlabel('C (regularization)', fontsize=13)
plt.ylabel('Accuracy', fontsize=13)
plt.title('SVM L1: Train vs Test Accuracy over C Range', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

best_idx_l1_range = int(np.argmax(test_scores_l1))
print(f"Best test accuracy (single split): {test_scores_l1[best_idx_l1_range]:.6f}")
print(f"Best C (single split)            : {C_range_l1[best_idx_l1_range]}")


### A-3. SVM L1 — Monte Carlo Hyperparameter Tuning for C

20 random train/test splits (different seeds) → mean ± variance accuracy at each C.  
Identical to the multiple-runs block in ML Notebook 3B Section D.


In [ ]:
# Monte Carlo: 20 trials, SVM L1
No_trials_l1 = 20
C_l1 = [1e-4, 1e-3, 0.1, 0.2, 0.4, 0.75, 1, 1.5, 3, 5, 10, 15, 20, 100, 300, 1000, 5000]

all_training_l1 = pd.DataFrame()
all_test_l1     = pd.DataFrame()

for seedN in range(1, No_trials_l1 + 1):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all, y_all, test_size=0.25, random_state=seedN)
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc  = sc.transform(X_te)

    train_acc = []
    test_acc  = []
    for alpha_run in C_l1:
        svm_l1 = LinearSVC(C=alpha_run, penalty="l1", loss='squared_hinge',
                           dual=False, max_iter=10000).fit(X_tr_sc, y_tr)
        train_acc.append(svm_l1.score(X_tr_sc, y_tr))
        test_acc.append(svm_l1.score(X_te_sc,  y_te))
    all_training_l1[seedN] = train_acc
    all_test_l1[seedN]     = test_acc

print(f"Monte Carlo complete: {No_trials_l1} trials x {len(C_l1)} C values")


In [ ]:
# Monte Carlo Graph — SVM L1
fig = plt.figure(figsize=(15, 6))
plt.xscale('log')
plt.errorbar(C_l1, all_training_l1.mean(axis=1),
             yerr=all_training_l1.var(axis=1),
             label='Training accuracy', marker='o', color='steelblue', capsize=4)
plt.errorbar(C_l1, all_test_l1.mean(axis=1),
             yerr=all_test_l1.var(axis=1),
             label='Test accuracy', marker='^', color='darkorange', capsize=4)
plt.ylabel('Accuracy', fontsize=13)
plt.xlabel('C', fontsize=13)
plt.title(f'SVM L1 — Monte Carlo Hyperparameter Tuning (n={No_trials_l1} trials)', fontsize=14)
plt.ylim(max(0, all_test_l1.mean(axis=1).min() - 0.05), 1.0)
plt.legend(fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Highest Test Set Achieved = {np.amax(all_test_l1.mean(axis=1)):.6f}")
print(f"Best C Parameter          = {C_l1[np.argmax(all_test_l1.mean(axis=1))]}")


### A-4. SVM L1 — Best C and Best Predictor Feature

In [ ]:
# Refit SVM L1 with best C from Monte Carlo
best_C_l1 = C_l1[np.argmax(all_test_l1.mean(axis=1))]

svm_l1_best = LinearSVC(C=best_C_l1, penalty="l1", loss='squared_hinge',
                         dual=False, max_iter=10000).fit(X_train_sc, y_train)

print(f"Best C from Monte Carlo : {best_C_l1}")
print(f"Training set score      : {svm_l1_best.score(X_train_sc, y_train):.6f}")
print(f"Test set score          : {svm_l1_best.score(X_test_sc,  y_test):.6f}")

# coef_ shape is (n_classes, n_features) for multi-class,
# or (1, n_features) for binary — take mean absolute value across classes
# then pick the feature with the highest average |coefficient|
coef_l1_matrix = svm_l1_best.coef_                          # shape: (n_classes, n_features)
coef_l1        = np.abs(coef_l1_matrix).mean(axis=0)       # shape: (n_features,)
top_idx_l1     = int(np.argmax(coef_l1))                   # always within feature_cols bounds

print(f"\nTop Predictor = {feature_cols[top_idx_l1]}")
print(f"Top Predictor Weight of ML with highest prediction = {coef_l1[top_idx_l1]:.6f}")


In [ ]:
# Coefficient plot — SVM L1, best C
fig = plt.figure(figsize=(14, 5))
x_pos = np.arange(len(feature_cols))

# coef_l1 is already mean |coef| per feature (from A-4 cell above)
raw_coef_l1 = svm_l1_best.coef_.mean(axis=0)   # signed mean for direction colour
plt.bar(x_pos, raw_coef_l1,
        color=['tomato' if v < 0 else 'steelblue' for v in raw_coef_l1], alpha=0.8)
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(x_pos, feature_cols, rotation=45, ha='right', fontsize=10)
plt.xlabel('Feature', fontsize=13)
plt.ylabel('Coefficient magnitude', fontsize=13)
plt.title(f'SVM L1 Coefficients — Best C = {best_C_l1}', fontsize=14)
plt.tight_layout()
plt.show()


---
# Part B — Support Vector Machine with L2 Regularization (SVM L2)

`LinearSVC(penalty="l2")` — the default penalty in scikit-learn.

Same four-step structure as Part A.


### B-1. SVM L2 — Single C (C = 1)

In [ ]:
# SVM L2 with C = 1
C_single_l2 = 1

svm_l2_single = LinearSVC(
    C=C_single_l2,
    penalty="l2",
    max_iter=10000
).fit(X_train_sc, y_train)

print(f"SVM L2  |  C = {C_single_l2}")
print(f"  Training set score : {svm_l2_single.score(X_train_sc, y_train):.6f}")
print(f"  Test set score     : {svm_l2_single.score(X_test_sc,  y_test):.6f}")


In [ ]:
# Classification Report — SVM L2, C = 1
y_pred_l2_single = svm_l2_single.predict(X_test_sc)
print("Classification Report (SVM L2, C=1):")
print(classification_report(y_test, y_pred_l2_single,
                             labels=unique_classes, target_names=target_names))


In [ ]:
# Confusion Matrix — SVM L2, C = 1
confmat_l2_single = confusion_matrix(y_true=y_test, y_pred=y_pred_l2_single,
                                     labels=unique_classes)

fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.matshow(confmat_l2_single, cmap=plt.cm.Blues, alpha=0.6)
for i in range(confmat_l2_single.shape[0]):
    for j in range(confmat_l2_single.shape[1]):
        ax.text(x=j, y=i, s=confmat_l2_single[i, j], va='center', ha='center', fontsize=14)
tick_labels = [''] + target_names
ax.set_xticklabels(tick_labels, rotation=30, ha='left')
ax.set_yticklabels(tick_labels)
ax.set_xlabel('Predicted label', fontsize=12)
ax.set_ylabel('True label', fontsize=12)
ax.set_title(f'Confusion Matrix — SVM L2  (C={C_single_l2})', fontsize=12, pad=12)
plt.tight_layout()
plt.show()


### B-2. SVM L2 — Range of C Values

In [ ]:
# SVM L2: scores over a range of C values (single split)
C_range_l2 = [1e-4, 1e-3, 0.1, 0.2, 0.4, 0.75, 1, 1.5, 3, 5, 10, 15, 20, 100, 300, 1000, 5000]

train_scores_l2 = []
test_scores_l2  = []

for c in C_range_l2:
    svm = LinearSVC(C=c, penalty="l2", max_iter=10000)
    svm.fit(X_train_sc, y_train)
    train_scores_l2.append(svm.score(X_train_sc, y_train))
    test_scores_l2.append(svm.score(X_test_sc,  y_test))
    print(f"C = {c:>8}  |  train: {train_scores_l2[-1]:.4f}  |  test: {test_scores_l2[-1]:.4f}")


In [ ]:
# Plot — SVM L2 range of C
fig = plt.figure(figsize=(12, 5))
plt.xscale('log')
plt.plot(C_range_l2, train_scores_l2, '-o', label='Training accuracy', color='steelblue')
plt.plot(C_range_l2, test_scores_l2,  '-^', label='Test accuracy',     color='darkorange')
plt.xlabel('C (regularization)', fontsize=13)
plt.ylabel('Accuracy', fontsize=13)
plt.title('SVM L2: Train vs Test Accuracy over C Range', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

best_idx_l2_range = int(np.argmax(test_scores_l2))
print(f"Best test accuracy (single split): {test_scores_l2[best_idx_l2_range]:.6f}")
print(f"Best C (single split)            : {C_range_l2[best_idx_l2_range]}")


### B-3. SVM L2 — Monte Carlo Hyperparameter Tuning for C

Same 20-trial Monte Carlo structure as Part A, using L2 penalty.


In [ ]:
# Monte Carlo: 20 trials, SVM L2
No_trials_l2 = 20
C_l2 = [1e-4, 1e-3, 0.1, 0.2, 0.4, 0.75, 1, 1.5, 3, 5, 10, 15, 20, 100, 300, 1000, 5000]

all_training_l2 = pd.DataFrame()
all_test_l2     = pd.DataFrame()

for seedN in range(1, No_trials_l2 + 1):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all, y_all, test_size=0.25, random_state=seedN)
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc  = sc.transform(X_te)

    train_acc = []
    test_acc  = []
    for alpha_run in C_l2:
        svm_l2 = LinearSVC(C=alpha_run, penalty="l2", max_iter=10000).fit(X_tr_sc, y_tr)
        train_acc.append(svm_l2.score(X_tr_sc, y_tr))
        test_acc.append(svm_l2.score(X_te_sc,  y_te))
    all_training_l2[seedN] = train_acc
    all_test_l2[seedN]     = test_acc

print(f"Monte Carlo complete: {No_trials_l2} trials x {len(C_l2)} C values")


In [ ]:
# Monte Carlo Graph SVM L2
fig = plt.figure(figsize=(15, 6))
plt.xscale('log')
plt.errorbar(C_l2, all_training_l2.mean(axis=1),
             yerr=all_training_l2.var(axis=1),
             label='Training accuracy', marker='o', color='steelblue', capsize=4)
plt.errorbar(C_l2, all_test_l2.mean(axis=1),
             yerr=all_test_l2.var(axis=1),
             label='Test accuracy', marker='^', color='darkorange', capsize=4)
plt.ylabel('Accuracy', fontsize=13)
plt.xlabel('C', fontsize=13)
plt.title(f'SVM L2 — Monte Carlo Hyperparameter Tuning (n={No_trials_l2} trials)', fontsize=14)
plt.ylim(max(0, all_test_l2.mean(axis=1).min() - 0.05), 1.0)
plt.legend(fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Highest Test Set Achieved = {np.amax(all_test_l2.mean(axis=1)):.6f}")
print(f"Best C Parameter          = {C_l2[np.argmax(all_test_l2.mean(axis=1))]}")


### B-4. SVM L2 — Best C and Best Predictor Feature

In [ ]:
# Refit SVM L2 with best C from Monte Carlo
best_C_l2 = C_l2[np.argmax(all_test_l2.mean(axis=1))]

svm_l2_best = LinearSVC(C=best_C_l2, penalty="l2",
                         max_iter=10000).fit(X_train_sc, y_train)

print(f"Best C from Monte Carlo : {best_C_l2}")
print(f"Training set score      : {svm_l2_best.score(X_train_sc, y_train):.6f}")
print(f"Test set score          : {svm_l2_best.score(X_test_sc,  y_test):.6f}")

# Same fix: average |coef| across classes, index stays within feature_cols
coef_l2_matrix = svm_l2_best.coef_
coef_l2        = np.abs(coef_l2_matrix).mean(axis=0)
top_idx_l2     = int(np.argmax(coef_l2))

print(f"\nTop Predictor = {feature_cols[top_idx_l2]}")
print(f"Top Predictor Weight of ML with highest prediction = {coef_l2[top_idx_l2]:.6f}")


In [ ]:
# Coefficient plot — SVM L2, best C
fig = plt.figure(figsize=(14, 5))
x_pos = np.arange(len(feature_cols))

raw_coef_l2 = svm_l2_best.coef_.mean(axis=0)
plt.bar(x_pos, raw_coef_l2,
        color=['tomato' if v < 0 else 'steelblue' for v in raw_coef_l2], alpha=0.8)
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(x_pos, feature_cols, rotation=45, ha='right', fontsize=10)
plt.xlabel('Feature', fontsize=13)
plt.ylabel('Coefficient magnitude', fontsize=13)
plt.title(f'SVM L2 Coefficients — Best C = {best_C_l2}', fontsize=14)
plt.tight_layout()
plt.show()


---
## Summary of Results

In [ ]:
summary = pd.DataFrame({
    'ML Method'    : ['Linear SVM (L1)', 'Linear SVM (L2)'],
    'Test Accuracy': [
        f"{np.amax(all_test_l1.mean(axis=1))*100:.2f}%",
        f"{np.amax(all_test_l2.mean(axis=1))*100:.2f}%"
    ],
    'Best C': [best_C_l1, best_C_l2],
    'Top Predictor': [feature_cols[top_idx_l1], feature_cols[top_idx_l2]]
})

print(summary.to_string(index=False))
